In [1]:
import pandas as pd
import json
import os
import numpy as np

In [ ]:
# Classe Humana - Escolha randômica de 3000 mil resumos de artigos do csv Corpus/metadados_completos_sol_sbc.csv ()
# Classe Polida por IAG - Escolha randômica de 3000 resumos de artigos do csv Corpus/metadados_completos_sol_sbc.csv) e enivar juto a PROMPT para API do ChatGPT para Reescrita
# Classe Gerada por IAG - Escolha randômica de 3000 artigos e após seleção usar título, introdução e conclusão na requisição via API para geração de um resumo do artigo

In [23]:
df = pd.read_csv('C:\\Users\\05646078199\\Projetos\\Projeto-Mestrado\\Corpus\\metadados_completos_sol_sbc.csv')

In [24]:
df.head()

,Title,Category,URL_Title,Authors,Event,Date,Box,Abstract,Keywords,Publisher,URL_Paper,index
0,A DAG-Based Post-Quantum Ledger,ANAIS DE EVENTO,https://sol.sbc.org.br/index.php/ladc_estendid...,"Freitas, Allan Edgard Silva",COMPANION PROCEEDINGS OF THE LATIN-AMERICAN SY...,2025-10-27,Caixa 1,Quantum computing threatens foundational crypt...,['No keywords available'],PDF (English),https://sol.sbc.org.br/index.php/ladc_estendid...,1
1,AI resources governance with OpenDID: Strategy...,ANAIS DE EVENTO,https://sol.sbc.org.br/index.php/ladc_estendid...,"Chacón, Lenin; Moraga, Kevin",COMPANION PROCEEDINGS OF THE LATIN-AMERICAN SY...,2025-10-27,Caixa 1,The accelerated adoption of artificial intelli...,"['Artificial Intelligence (AI)', 'Decentralize...",PDF (English),https://sol.sbc.org.br/index.php/ladc_estendid...,2
2,Digital Academic Certification with Blockchain...,ANAIS DE EVENTO,https://sol.sbc.org.br/index.php/ladc_estendid...,"Blanco, Pablo; Betarte, Gustavo; Luna, Carlos;...",COMPANION PROCEEDINGS OF THE LATIN-AMERICAN SY...,2025-10-27,Caixa 1,Academic certificates are essential credential...,"['Academic Certification', 'Blockchain', 'Hype...",PDF (English),https://sol.sbc.org.br/index.php/ladc_estendid...,3
3,An API-Driven Framework for Performance Testin...,ANAIS DE EVENTO,https://sol.sbc.org.br/index.php/ladc_estendid...,"Cardoso, Carlos; Silva, Caio; Veloso, Alan; So...",COMPANION PROCEEDINGS OF THE LATIN-AMERICAN SY...,2025-10-27,Caixa 1,As Distributed Ledger Technologies (DLTs) matu...,"['Performance Testing', 'Blockchain', 'Apache ...",PDF (English),https://sol.sbc.org.br/index.php/ladc_estendid...,4
4,Enhancing Data Provenance in mHealth: An Archi...,ANAIS DE EVENTO,https://sol.sbc.org.br/index.php/ladc_estendid...,"Velasco, Gislainy Crisostomo; Vaz, Noeli Antôn...",COMPANION PROCEEDINGS OF THE LATIN-AMERICAN SY...,2025-10-27,Caixa 1,The advancement of digitalization in healthcar...,"['Data Provenance', 'W3C PROV', 'mHealth', 'Bl...",PDF (English),https://sol.sbc.org.br/index.php/ladc_estendid...,5


In [25]:
# Mapear o idioma dos artigos, inserindo-os em uma coluna no df

json_folder = 'C:\\Users\\05646078199\\Projetos\\Projeto-Mestrado\\Corpus\\json' 

# ============================================================
# COLUNAS NOVAS
# ============================================================

df["Paper_Language"] = None

df["has_intro"] = 0
df["has_conclusion"] = 0
df["flag_intro_conc"] = 0


# ============================================================
# PADRÕES
# ============================================================

INTRO_PATTERNS = [
    "introd"
]

CONCLUSION_PATTERNS = [
    "concl",
    "considerações finais",
    "consideracoes finais",
    "final remarks",
    "concluding"
]


# ============================================================
# PROCESSAR JSONS
# ============================================================

for file_name in os.listdir(json_folder):

    if file_name.endswith(".json"):

        try:

            # ====================================================
            # EXTRAIR ÍNDICE
            # ====================================================

            json_index = int(
                file_name.replace("sbc_", "").replace(".json", "")
            )

            print(f"\nProcessando {file_name}")
            print(f"Índice JSON: {json_index}")

            df_index = json_index

            # ====================================================
            # ABRIR JSON
            # ====================================================

            file_path = os.path.join(json_folder, file_name)

            with open(file_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # ====================================================
            # PEGAR IDIOMA
            # ====================================================

            language = data.get("language", None)

            print(f"Idioma encontrado: {language}")

            df.loc[df_index, "Paper_Language"] = language

            # ====================================================
            # EXTRAIR TODOS OS TITLES
            # ====================================================

            titles = []

            def extract_titles(obj):

                if isinstance(obj, dict):

                    for key, value in obj.items():

                        # Encontrou chave "title"
                        if key == "title":

                            if isinstance(value, str):

                                titles.append(value.lower())

                        # Busca recursiva
                        extract_titles(value)

                elif isinstance(obj, list):

                    for item in obj:
                        extract_titles(item)

            extract_titles(data)

            # ====================================================
            # DETECTAR INTRODUÇÃO
            # ====================================================

            has_intro = any(
                any(pattern in title for pattern in INTRO_PATTERNS)
                for title in titles
            )

            # ====================================================
            # DETECTAR CONCLUSÃO
            # ====================================================

            has_conclusion = any(
                any(pattern in title for pattern in CONCLUSION_PATTERNS)
                for title in titles
            )

            # ====================================================
            # SALVAR FLAGS
            # ====================================================

            df.loc[df_index, "has_intro"] = int(has_intro)

            df.loc[df_index, "has_conclusion"] = int(has_conclusion)

            df.loc[df_index, "flag_intro_conc"] = int(
                has_intro and has_conclusion
            )

            # ====================================================
            # DEBUG
            # ====================================================

            print(f"Has Intro: {has_intro}")
            print(f"Has Conclusion: {has_conclusion}")

        except Exception as e:

            print(f"Erro ao processar {file_name}: {e}")


# ============================================================
# RESUMO FINAL
# ============================================================

print("\n")
print("=" * 60)
print("RESUMO")
print("=" * 60)

print(
    df[
        [
            "has_intro",
            "has_conclusion",
            "flag_intro_conc"
        ]
    ].sum()
)


Processando sbc_0.json
Índice JSON: 0
Idioma encontrado: en
Has Intro: True
Has Conclusion: True

Processando sbc_1.json
Índice JSON: 1
Idioma encontrado: en
Has Intro: True
Has Conclusion: True

Processando sbc_100.json
Índice JSON: 100
Idioma encontrado: en
Has Intro: True
Has Conclusion: True

Processando sbc_1000.json
Índice JSON: 1000
Idioma encontrado: en
Has Intro: True
Has Conclusion: True

Processando sbc_10000.json
Índice JSON: 10000
Idioma encontrado: en
Has Intro: True
Has Conclusion: True

Processando sbc_10001.json
Índice JSON: 10001
Idioma encontrado: en
Has Intro: True
Has Conclusion: True

Processando sbc_10002.json
Índice JSON: 10002
Idioma encontrado: en
Has Intro: True
Has Conclusion: True

Processando sbc_10003.json
Índice JSON: 10003
Idioma encontrado: en
Has Intro: True
Has Conclusion: False

Processando sbc_10004.json
Índice JSON: 10004
Idioma encontrado: en
Has Intro: True
Has Conclusion: True

Processando sbc_10005.json
Índice JSON: 10005
Idioma encontrado: e

In [27]:
df.head()

,Title,Category,URL_Title,Authors,Event,Date,Box,Abstract,Keywords,Publisher,URL_Paper,index,Paper_Language,has_intro,has_conclusion,flag_intro_conc
0,A DAG-Based Post-Quantum Ledger,ANAIS DE EVENTO,https://sol.sbc.org.br/index.php/ladc_estendid...,"Freitas, Allan Edgard Silva",COMPANION PROCEEDINGS OF THE LATIN-AMERICAN SY...,2025-10-27,Caixa 1,Quantum computing threatens foundational crypt...,['No keywords available'],PDF (English),https://sol.sbc.org.br/index.php/ladc_estendid...,1,en,1,1,1
1,AI resources governance with OpenDID: Strategy...,ANAIS DE EVENTO,https://sol.sbc.org.br/index.php/ladc_estendid...,"Chacón, Lenin; Moraga, Kevin",COMPANION PROCEEDINGS OF THE LATIN-AMERICAN SY...,2025-10-27,Caixa 1,The accelerated adoption of artificial intelli...,"['Artificial Intelligence (AI)', 'Decentralize...",PDF (English),https://sol.sbc.org.br/index.php/ladc_estendid...,2,en,1,1,1
2,Digital Academic Certification with Blockchain...,ANAIS DE EVENTO,https://sol.sbc.org.br/index.php/ladc_estendid...,"Blanco, Pablo; Betarte, Gustavo; Luna, Carlos;...",COMPANION PROCEEDINGS OF THE LATIN-AMERICAN SY...,2025-10-27,Caixa 1,Academic certificates are essential credential...,"['Academic Certification', 'Blockchain', 'Hype...",PDF (English),https://sol.sbc.org.br/index.php/ladc_estendid...,3,en,1,1,1
3,An API-Driven Framework for Performance Testin...,ANAIS DE EVENTO,https://sol.sbc.org.br/index.php/ladc_estendid...,"Cardoso, Carlos; Silva, Caio; Veloso, Alan; So...",COMPANION PROCEEDINGS OF THE LATIN-AMERICAN SY...,2025-10-27,Caixa 1,As Distributed Ledger Technologies (DLTs) matu...,"['Performance Testing', 'Blockchain', 'Apache ...",PDF (English),https://sol.sbc.org.br/index.php/ladc_estendid...,4,en,1,1,1
4,Enhancing Data Provenance in mHealth: An Archi...,ANAIS DE EVENTO,https://sol.sbc.org.br/index.php/ladc_estendid...,"Velasco, Gislainy Crisostomo; Vaz, Noeli Antôn...",COMPANION PROCEEDINGS OF THE LATIN-AMERICAN SY...,2025-10-27,Caixa 1,The advancement of digitalization in healthcar...,"['Data Provenance', 'W3C PROV', 'mHealth', 'Bl...",PDF (English),https://sol.sbc.org.br/index.php/ladc_estendid...,5,en,1,1,1


In [28]:
df.to_csv('C:\\Users\\05646078199\\Projetos\\Projeto-Mestrado\\Corpus\\metadados_completos_sol_sbc_com_idioma_artigo.csv', index=False)

In [29]:
import re

AREA_MAP = {
    # IA
    r'ENIAC|BRACIS|KDMILE|STIL|BWAIF|ARTIFICIAL INTELLIGENCE|MACHINE LEARNING|ERAMIA|WESAAC':
        'IA',

    # IHC
    r'\bIHC\b|HUMAN-COMPUTER INTERACTION|INTERACTIVE SYSTEMS|CAPAIHC|WEIHC|WAIHCWS|WIDE':
        'IHC',

    # Engenharia de Software
    r'SBES|SBCARS|SAST|VEM|MSSIS|SE4FP|SE4GAMES|SEDT|ISE|WBOTS|SOFTWARE ENGINEERING|OPENSCIENSE|CIBSE|WASHES|BWARE|CBSOFT|SBQS':
        'ES',

    # Redes
    r'SBRC|ERRC|WGRS|WPERFORMANCE|WPEIF|W6G|WQUNETS|WSLICE|WFIBRE|WPIETF|NETWORK|NETWORKS|INTERNET SERVICES':
        'REDES',

    # Segurança
    r'SBSEG|LADC|CYBERSECURITY|FORENSICS|DEPENDABILITY|SAFETY|SECURITY|WTF|RECS|WSENSING|SAFELIFE|WAFERS|STAMP|DIGITAL IDENTITY':
        'SEG',

    # Banco de Dados
    r'SBBD|ERBD|DATA MANAGEMENT|DATABASE':
        'BD',

    # Alto Desempenho
    r'SSCAD|SBAC-PAD|HIGH PERFORMANCE|ERAD':
        'HPC',

    # Educação
    r'CBIE|SBIE|WIE|WEI|EDUCOMP|CTRL\+E|SBC-EB|WAPLA|WAVE|WPCI|WEADEH|WETIE|DESAFIE|WIEI|EDUCATION|EDUCA':
        'EDU',

    # Sistemas de Informação
    r'SBSI|WSIS|INFORMATION SYSTEMS|ISYS':
        'SI',

    # Computação Gráfica / Multimídia
    r'SIBGRAPI|SVR|WVC|WEBMEDIA|IMX|SENSORYX|XR IN GAMES|COMPUTER GRAPHICS|MULTIMEDIA|VIRTUAL|AUGMENTED REALITY|SBCM':
        'CG',

    # Arquitetura / Hardware
    r'SBCCI|SBESC|COMPUTER ARCHITECTURE|INTEGRATED CIRCUITS':
        'ARQ',

    # Teoria da Computação
    r'\bETC\b|WEIT|WBL|SBLP|SBMF|THEORY OF COMPUTATION|FORMAL METHODS|LOGIC':
        'TC',

    # Robótica
    r'SBR/LARS|ROBOTICS':
        'ROB',

    # Saúde
    r'SBCAS|BSB|HEALTH|BIOINFORMATICS|ERCAS':
        'SAUDE',

    # Ubicomp / IoT
    r'SBCUP|UBIQUITOUS|PERVASIVE|WBCI|URBAN COMPUTING|WCGA':
        'UBICOMP',

    # Social / Ética
    r'WIT|WICS|MOSAICO|ETHICS|INCLUSION|DIVERSITY|SOCIAL':
        'SOC',

    # Quântica
    r'QUANTUM|QUNETS':
        'QUANT',

    # Jogos
    r'SBGAMES|WIPLAY|XR IN GAMES|GAME':
        'GAMES',
}


def categorize_event(event_name: str) -> str:
    """
    Categoriza eventos da SBC em macroáreas.
    """

    event_upper = event_name.upper()

    for pattern, area in AREA_MAP.items():
        if re.search(pattern, event_upper):
            return area

    return 'GERAL'




df["Area"] = df["Event"].apply(categorize_event)

In [30]:
df.to_csv('C:\\Users\\05646078199\\Projetos\\Projeto-Mestrado\\Corpus\\metadados_completos_sol_sbc_com_idioma_e_area.csv', index=False)

In [35]:
df = pd.read_csv('C:\\Users\\05646078199\\Projetos\\Projeto-Mestrado\\Corpus\\metadados_completos_sol_sbc_com_idioma_e_area.csv')

In [36]:
# ============================================================
# IMPORTS
# ============================================================

import os
import json
import pandas as pd


# ============================================================
# CONFIGURAÇÕES
# ============================================================

TARGET_PER_CLASS = 2450

RANDOM_STATE = 42

json_folder = r'C:\Users\05646078199\Projetos\Projeto-Mestrado\Corpus\json'


# ============================================================
# ETAPA 1 — PREPARAÇÃO DO DATAFRAME
# ============================================================

# Converter coluna Date
df['Date'] = pd.to_datetime(df['Date'])

# Extrair ano
df['Year'] = df['Date'].dt.year


# ============================================================
# ETAPA 2 — CRIAR COLUNAS AUXILIARES
# ============================================================

df["Paper_Language"] = None

df["has_intro"] = 0
df["has_conclusion"] = 0
df["flag_intro_conc"] = 0


# ============================================================
# ETAPA 3 — PADRÕES DE DETECÇÃO
# ============================================================

INTRO_PATTERNS = [
    "introd"
]

CONCLUSION_PATTERNS = [
    "concl",
    "considerações finais",
    "consideracoes finais",
    "final remarks",
    "concluding"
]


# ============================================================
# ETAPA 4 — EXTRAÇÃO RECURSIVA DE TITLES
# ============================================================

def extract_titles(obj, titles=None):

    if titles is None:
        titles = []

    if isinstance(obj, dict):

        for key, value in obj.items():

            if key == "title":

                if isinstance(value, str):

                    titles.append(value.lower())

            extract_titles(value, titles)

    elif isinstance(obj, list):

        for item in obj:

            extract_titles(item, titles)

    return titles


# ============================================================
# ETAPA 5 — MAPEAR IDIOMA + FLAGS ESTRUTURAIS
# ============================================================

print("=" * 60)
print("PROCESSANDO JSONS")
print("=" * 60)

for file_name in os.listdir(json_folder):

    if file_name.endswith(".json"):

        try:

            # ------------------------------------------------
            # EXTRAIR ÍNDICE
            # ------------------------------------------------

            json_index = int(
                file_name.replace("sbc_", "").replace(".json", "")
            )

            print(f"\nProcessando: {file_name}")

            df_index = json_index

            # ------------------------------------------------
            # ABRIR JSON
            # ------------------------------------------------

            file_path = os.path.join(json_folder, file_name)

            with open(file_path, "r", encoding="utf-8") as f:

                data = json.load(f)

            # ------------------------------------------------
            # IDIOMA
            # ------------------------------------------------

            language = data.get("language", None)

            df.loc[df_index, "Paper_Language"] = language

            # ------------------------------------------------
            # TITLES
            # ------------------------------------------------

            titles = extract_titles(data)

            # ------------------------------------------------
            # DETECTAR INTRODUÇÃO
            # ------------------------------------------------

            has_intro = any(
                any(pattern in title for pattern in INTRO_PATTERNS)
                for title in titles
            )

            # ------------------------------------------------
            # DETECTAR CONCLUSÃO
            # ------------------------------------------------

            has_conclusion = any(
                any(pattern in title for pattern in CONCLUSION_PATTERNS)
                for title in titles
            )

            # ------------------------------------------------
            # FLAGS
            # ------------------------------------------------

            df.loc[df_index, "has_intro"] = int(has_intro)

            df.loc[df_index, "has_conclusion"] = int(has_conclusion)

            df.loc[df_index, "flag_intro_conc"] = int(
                has_intro and has_conclusion
            )

        except Exception as e:

            print(f"Erro ao processar {file_name}: {e}")


# ============================================================
# ETAPA 6 — FILTRAR CORPUS FINAL
# ============================================================

df_filtered = df[
    (df['Year'] >= 2010) &
    (df['Year'] <= 2022) &
    (df['Paper_Language'] == 'pt') &
    (df['flag_intro_conc'] == 1)
].copy()


# ============================================================
# ETAPA 7 — VISÃO GERAL DO CORPUS
# ============================================================

print("\n")
print("=" * 60)
print("DISTRIBUIÇÃO DE ARTIGOS POR ANO E ÁREA")
print("=" * 60)

for year in sorted(df_filtered['Year'].dropna().unique()):

    df_year = df_filtered[df_filtered['Year'] == year]

    for area in sorted(df_year['Area'].dropna().unique()):

        count_area = df_year[
            df_year['Area'] == area
        ].shape[0]

        print(
            f"Ano: {year} | "
            f"Área: {area} | "
            f"Quantidade: {count_area}"
        )

    count_year = df_year.shape[0]

    print(f"Ano: {year} | Total: {count_year}")

    print("-" * 60)


# ============================================================
# ETAPA 8 — DISTRIBUIÇÃO TEMPORAL
# ============================================================

year_distribution = (
    df_filtered
    .groupby('Year')
    .size()
    .reset_index(name='Count')
)

year_distribution['Year_Percentage'] = (
    year_distribution['Count'] /
    year_distribution['Count'].sum()
)

year_distribution['Target_Total'] = (
    year_distribution['Year_Percentage'] *
    (TARGET_PER_CLASS * 3)
).round().astype(int)

print("\n")
print("=" * 60)
print("DISTRIBUIÇÃO TEMPORAL")
print("=" * 60)

print(year_distribution)


# ============================================================
# ETAPA 9 — DISTRIBUIÇÃO POR ÁREA
# ============================================================

area_distribution = (
    df_filtered
    .groupby(['Year', 'Area'])
    .size()
    .reset_index(name='Count')
)

year_totals = (
    area_distribution
    .groupby('Year')['Count']
    .sum()
    .reset_index(name='Year_Total')
)

area_distribution = area_distribution.merge(
    year_totals,
    on='Year'
)

area_distribution['Area_Percentage'] = (
    area_distribution['Count'] /
    area_distribution['Year_Total']
)

area_distribution = area_distribution.merge(
    year_distribution[['Year', 'Target_Total']],
    on='Year'
)

area_distribution['Target_Stratum'] = (
    area_distribution['Area_Percentage'] *
    area_distribution['Target_Total']
).round().astype(int)

print("\n")
print("=" * 60)
print("DISTRIBUIÇÃO POR ÁREA")
print("=" * 60)

print(area_distribution)


# ============================================================
# ETAPA 10 — DIVISÃO ENTRE CLASSES
# ============================================================

area_distribution['Per_Class'] = (
    area_distribution['Target_Stratum'] // 3
)

print("\n")
print("=" * 60)
print("DIVISÃO ENTRE CLASSES")
print("=" * 60)

print(
    area_distribution[
        [
            'Year',
            'Area',
            'Target_Stratum',
            'Per_Class'
        ]
    ]
)


# ============================================================
# ETAPA 11 — AMOSTRAGEM DOS PAPERS
# ============================================================

selected_rows = []

classes = [
    'Humana',
    'Polida_IA',
    'Gerada'
]

for _, row in area_distribution.iterrows():

    year = row['Year']

    area = row['Area']

    per_class = row['Per_Class']

    # --------------------------------------------------------
    # FILTRAR CANDIDATOS
    # --------------------------------------------------------

    candidates = df_filtered[
        (df_filtered['Year'] == year) &
        (df_filtered['Area'] == area)
    ].copy()

    # --------------------------------------------------------
    # EMBARALHAR
    # --------------------------------------------------------

    candidates = candidates.sample(
        frac=1,
        random_state=RANDOM_STATE
    )

    # --------------------------------------------------------
    # TOTAL NECESSÁRIO
    # --------------------------------------------------------

    total_needed = per_class * 3

    total_needed = min(
        total_needed,
        len(candidates)
    )

    # --------------------------------------------------------
    # RECORTE
    # --------------------------------------------------------

    candidates = candidates.iloc[:total_needed]

    # --------------------------------------------------------
    # DIVISÃO ENTRE CLASSES
    # --------------------------------------------------------

    split_size = len(candidates) // 3

    splits = [
        candidates.iloc[:split_size],
        candidates.iloc[split_size:split_size * 2],
        candidates.iloc[split_size * 2:]
    ]

    # --------------------------------------------------------
    # ATRIBUIR CLASSES
    # --------------------------------------------------------

    for class_name, split_df in zip(classes, splits):

        split_df = split_df.copy()

        split_df['Class'] = class_name

        selected_rows.append(split_df)


# ============================================================
# ETAPA 12 — DATAFRAME FINAL
# ============================================================

df_final = pd.concat(selected_rows)

df_final = df_final.reset_index(drop=True)

# Novas colunas
df_final['Introduction'] = None
df_final['Conclusion'] = None


# ============================================================
# ETAPA 13 — EXTRAÇÃO DE INTRODUÇÃO E CONCLUSÃO
# ============================================================

def extract_sections(obj, current_title=None, sections=None):

    if sections is None:
        sections = []

    if isinstance(obj, dict):

        title = obj.get("title", current_title)

        text_parts = []

        for key in ["text", "content", "paragraphs"]:

            if key in obj:

                value = obj[key]

                if isinstance(value, str):

                    text_parts.append(value)

                elif isinstance(value, list):

                    for item in value:

                        if isinstance(item, str):

                            text_parts.append(item)

                        elif isinstance(item, dict):

                            if "text" in item:

                                text_parts.append(
                                    str(item["text"])
                                )

        if title and text_parts:

            sections.append({
                "title": str(title).lower(),
                "text": "\n".join(text_parts)
            })

        for value in obj.values():

            extract_sections(
                value,
                title,
                sections
            )

    elif isinstance(obj, list):

        for item in obj:

            extract_sections(
                item,
                current_title,
                sections
            )

    return sections


print("\n")
print("=" * 60)
print("EXTRAINDO INTRODUÇÃO E CONCLUSÃO")
print("=" * 60)

for idx, row in df_final.iterrows():

    paper_id = row['index']

    file_name = f"sbc_{paper_id}.json"

    file_path = os.path.join(
        json_folder,
        file_name
    )

    if not os.path.exists(file_path):

        print(f"JSON não encontrado: {file_name}")

        continue

    try:

        with open(file_path, "r", encoding="utf-8") as f:

            data = json.load(f)

        sections = extract_sections(data)

        introduction_text = None

        conclusion_text = None

        # ----------------------------------------------------
        # BUSCAR SEÇÕES
        # ----------------------------------------------------

        for section in sections:

            title = section['title']

            text = section['text']

            # INTRODUÇÃO
            if introduction_text is None:

                if any(
                    pattern in title
                    for pattern in INTRO_PATTERNS
                ):

                    introduction_text = text

            # CONCLUSÃO
            if conclusion_text is None:

                if any(
                    pattern in title
                    for pattern in CONCLUSION_PATTERNS
                ):

                    conclusion_text = text

        # ----------------------------------------------------
        # SALVAR
        # ----------------------------------------------------

        df_final.at[idx, 'Introduction'] = introduction_text

        df_final.at[idx, 'Conclusion'] = conclusion_text

        print(f"Processado: {paper_id}")

    except Exception as e:

        print(f"Erro em {file_name}: {e}")


# ============================================================
# ETAPA 14 — SELECIONAR COLUNAS FINAIS
# ============================================================

df_final = df_final[
    [
        'index',
        'Title',
        'Year',
        'Event',
        'Area',
        'Abstract',
        'Introduction',
        'Conclusion',
        'Class'
    ]
]


# ============================================================
# ETAPA 15 — VERIFICAÇÕES FINAIS
# ============================================================

print("\n")
print("=" * 60)
print("VERIFICAÇÕES FINAIS")
print("=" * 60)

print("\nQuantidade por classe:")
print(df_final['Class'].value_counts())

print("\nQuantidade total:")
print(df_final.shape[0])

print("\nPapers duplicados:")
print(df_final['index'].duplicated().sum())

print("\nIntroduções ausentes:")
print(df_final['Introduction'].isna().sum())

print("\nConclusões ausentes:")
print(df_final['Conclusion'].isna().sum())


# ============================================================
# ETAPA 16 — EXPORTAÇÃO
# ============================================================

df_final.to_csv(
    'dataset_control.csv',
    index=False
)

print("\n")
print("=" * 60)
print("DATAFRAME EXPORTADO COM SUCESSO")
print("=" * 60)

PROCESSANDO JSONS

Processando: sbc_0.json

Processando: sbc_1.json

Processando: sbc_100.json

Processando: sbc_1000.json

Processando: sbc_10000.json

Processando: sbc_10001.json

Processando: sbc_10002.json

Processando: sbc_10003.json

Processando: sbc_10004.json

Processando: sbc_10005.json

Processando: sbc_10006.json

Processando: sbc_10007.json

Processando: sbc_10008.json

Processando: sbc_10009.json

Processando: sbc_1001.json

Processando: sbc_10010.json

Processando: sbc_10011.json

Processando: sbc_10012.json

Processando: sbc_10013.json

Processando: sbc_10014.json

Processando: sbc_10015.json

Processando: sbc_10016.json

Processando: sbc_10017.json

Processando: sbc_10018.json

Processando: sbc_10019.json

Processando: sbc_1002.json

Processando: sbc_10020.json

Processando: sbc_10021.json

Processando: sbc_10022.json

Processando: sbc_10023.json

Processando: sbc_10024.json

Processando: sbc_10025.json

Processando: sbc_10026.json

Processando: sbc_10027.json

Processa

### Delimitação de Seções

In [1]:
# ============================================================
# ETAPA 1 — IMPORTS
# ============================================================

import os
import json
import pandas as pd
from collections import Counter


# ============================================================
# ETAPA 2 — DIRETÓRIO DOS JSONS
# ============================================================

# Pasta onde estão os arquivos JSON
JSON_FOLDER = r'C:\Users\05646078199\Projetos\Projeto-Mestrado\Corpus\json'


# ============================================================
# ETAPA 3 — LISTA DE TÍTULOS DE SEÇÕES
# ============================================================

all_titles = []


# ============================================================
# ETAPA 4 — PERCORRER TODOS OS JSONS
# ============================================================

for file_name in os.listdir(JSON_FOLDER):

    # Apenas arquivos .json
    if not file_name.endswith('.json'):
        continue

    file_path = os.path.join(JSON_FOLDER, file_name)

    try:

        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # ====================================================
        # Procurar todas as ocorrências de "title"
        # ====================================================

        def extract_titles(obj):

            if isinstance(obj, dict):

                for key, value in obj.items():

                    # Se encontrou chave "title"
                    if key == 'title':

                        if isinstance(value, str):

                            all_titles.append(value)

                    # Continuar busca recursiva
                    extract_titles(value)

            elif isinstance(obj, list):

                for item in obj:
                    extract_titles(item)

        extract_titles(data)

    except Exception as e:

        print(f'Erro ao processar {file_name}: {e}')


# ============================================================
# ETAPA 5 — CONTAGEM DE FREQUÊNCIA
# ============================================================

title_counter = Counter(all_titles)


# ============================================================
# ETAPA 6 — DATAFRAME FINAL
# ============================================================

df_sections = pd.DataFrame(
    title_counter.items(),
    columns=['Section_Title', 'Frequency']
)

# Ordenar por frequência
df_sections = df_sections.sort_values(
    by='Frequency',
    ascending=False
)

# Resetar índice
df_sections = df_sections.reset_index(drop=True)


# ============================================================
# ETAPA 7 — EXIBIR RESULTADOS
# ============================================================

print(df_sections)


# ============================================================
# ETAPA 8 — EXPORTAR CSV
# ============================================================

df_sections.to_csv(
    'section_titles_frequency.csv',
    index=False,
    encoding='utf-8-sig'
)

print('\nCSV exportado com sucesso.')

                                            Section_Title  Frequency
0                                            Referˆencias       7101
1                                         1. Introduc¸˜ao       6885
2                                           1. Introdução       5916
3                                             Referências       5523
4                                              References       4632
...                                                   ...        ...
185935       5.2 Months 3–6: Institutional Authentication          1
185936  5.1 Months 1–2: Authentication for access to A...          1
185937                           5 IMPLEMENTATION ROADMAP          1
185938                      4 ETHICAL AND SOCIAL ARGUMENT          1
185939                        2.2.2. Cultural Differences          1

[185940 rows x 2 columns]

CSV exportado com sucesso.


In [3]:
df_sections.head(50)

,Section_Title,Frequency
0,Referˆencias,7101
1,1. Introduc¸˜ao,6885
2,1. Introdução,5916
3,Referências,5523
4,References,4632
5,1. Introduction,3662
6,##,3343
7,2. Trabalhos Relacionados,2797
8,Agradecimentos,2492
9,REFERENCES,1765
